## 2. Imports & Configuration

In [1]:
import os
import json
import warnings
import textwrap
from typing import TypedDict, Annotated, List, Optional, Any
from io import StringIO

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate

from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages

warnings.filterwarnings("ignore")

# ── LLM Setup ─────────────────────────────────────────────────────────────────
# Set your API key:  os.environ["OPENAI_API_KEY"] = "sk-..."
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "YOUR_KEY_HERE")

llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0,
    api_key=OPENAI_API_KEY
)

print("LLM configured:", llm.model_name)


LLM configured: gpt-4o


## 3. Agent State — The Shared Memory of the Graph

In [ ]:
class AnalystState(TypedDict):
    """
    Shared state passed between all agents in the LangGraph graph.
    Every agent reads from and writes to this state.
    """
    # ── Input ──────────────────────────────────────────────────────────────────
    user_query: str                        # Natural language question from user
    raw_data: Optional[pd.DataFrame]       # Original uploaded dataframe
    file_path: Optional[str]               # Path to CSV/Excel (if file-based)

    # ── Intermediate State ─────────────────────────────────────────────────────
    schema_info: Optional[dict]            # Output of Schema Analysis Agent
    cleaned_data: Optional[pd.DataFrame]  # Output of Data Cleaning Agent
    query_plan: Optional[str]             # Output of Query Planning Agent (code)
    analysis_results: Optional[dict]      # Output of Statistical Analysis Agent
    charts: Optional[list]                # Output of Visualization Agent (Plotly figs)

    # ── Control Flow ───────────────────────────────────────────────────────────
    next_agent: Optional[str]             # Supervisor routing decision
    errors: List[str]                     # Accumulated errors across agents
    agent_logs: List[str]                 # Step-by-step execution trace

    # ── Output ─────────────────────────────────────────────────────────────────
    final_report: Optional[str]           # Markdown report from Report Agent

print("AnalystState TypedDict defined ")
print("Fields:", list(AnalystState.__annotations__.keys()))


## 4. Load Titanic Dataset

In [ ]:
# Load Titanic from seaborn's hosted CSV (no file needed)
titanic_url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df_titanic = pd.read_csv(titanic_url)

print(f"Loaded Titanic dataset: {df_titanic.shape[0]} rows × {df_titanic.shape[1]} columns")
df_titanic.head(3)


## 5. Agent 1 — Schema Analysis Agent
Infers column types, null rates, cardinality, and data quality summary.
Uses GPT-4o to classify each column's analytical role.


In [ ]:
def schema_analysis_agent(state: AnalystState) -> AnalystState:
    """
    Analyzes the dataframe structure and enriches it with LLM-inferred metadata.
    
    Responsibilities:
    - Compute null %, unique counts, dtype for every column
    - Ask LLM to classify each column (categorical/numerical/temporal/id/target)
    - Identify likely target variable from user query context
    - Store structured schema_info in state
    """
    log = "🔍 [Schema Agent] Starting schema analysis..."
    print(log)
    
    df: pd.DataFrame = state["raw_data"]
    user_query: str   = state["user_query"]
    
    # ── Step 1: Compute raw statistics ────────────────────────────────────────
    schema_stats = {}
    for col in df.columns:
        schema_stats[col] = {
            "dtype":        str(df[col].dtype),
            "null_pct":     round(df[col].isnull().mean() * 100, 2),
            "unique_count": int(df[col].nunique()),
            "sample_values": df[col].dropna().head(3).tolist()
        }
    
    # ── Step 2: LLM classifies each column ────────────────────────────────────
    schema_json_str = json.dumps(schema_stats, indent=2, default=str)
    
    system_prompt = textwrap.dedent("""
        You are a data schema analyst. Given column statistics from a DataFrame,
        return a JSON object where each key is a column name and the value is:
        {
          "role": one of [categorical, numerical, temporal, id, target, text],
          "description": "brief one-line description of what this column likely represents",
          "analytical_importance": one of [high, medium, low]
        }
        Return ONLY valid JSON. No markdown, no explanation.
    """)
    
    user_prompt = f"""
        User's question: {user_query}
        
        Column statistics:
        {schema_json_str}
    """
    
    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt)
    ])
    
    # ── Step 3: Parse LLM response ────────────────────────────────────────────
    try:
        llm_classifications = json.loads(response.content)
    except json.JSONDecodeError:
        # Graceful fallback: extract JSON from response
        content = response.content
        start, end = content.find("{"), content.rfind("}") + 1
        llm_classifications = json.loads(content[start:end]) if start != -1 else {}
    
    # ── Step 4: Merge stats + LLM classification ──────────────────────────────
    schema_info = {
        "shape":           {"rows": df.shape[0], "cols": df.shape[1]},
        "columns":         {},
        "high_null_cols":  [],
        "target_hint":     None
    }
    
    for col in df.columns:
        schema_info["columns"][col] = {
            **schema_stats[col],
            **(llm_classifications.get(col, {}))
        }
        if schema_stats[col]["null_pct"] > 20:
            schema_info["high_null_cols"].append(col)
        if llm_classifications.get(col, {}).get("role") == "target":
            schema_info["target_hint"] = col
    
    log2 = (f" [Schema Agent] Done. "
            f"Shape: {df.shape}, "
            f"High-null cols: {schema_info['high_null_cols']}, "
            f"Target hint: {schema_info['target_hint']}")
    print(log2)
    
    return {
        **state,
        "schema_info": schema_info,
        "agent_logs": state.get("agent_logs", []) + [log, log2]
    }

print("schema_analysis_agent defined ")


## 6. Agent 2 — Data Cleaning Agent
Handles nulls, outliers, and type casting based on schema_info.
LLM generates a cleaning strategy; agent executes it.


In [ ]:
def data_cleaning_agent(state: AnalystState) -> AnalystState:
    """
    Cleans the dataframe based on schema_info.
    
    Responsibilities:
    - Fill or drop nulls based on column role & null_pct
    - Cast columns to correct dtypes
    - Remove or cap outliers in numerical columns
    - Generate a human-readable cleaning log
    """
    log = "🧹 [Cleaning Agent] Starting data cleaning..."
    print(log)
    
    df: pd.DataFrame     = state["raw_data"].copy()
    schema_info: dict    = state["schema_info"]
    
    cleaning_steps = []
    
    for col, meta in schema_info["columns"].items():
        null_pct = meta["null_pct"]
        role     = meta.get("role", "unknown")
        dtype    = meta["dtype"]
        
        # ── Null handling ──────────────────────────────────────────────────────
        if null_pct > 50:
            df.drop(columns=[col], inplace=True)
            cleaning_steps.append(f"Dropped '{col}' — {null_pct}% nulls (>50% threshold)")
            continue
        
        if col not in df.columns:
            continue
            
        if null_pct > 0:
            if role == "numerical" or dtype in ["float64", "int64", "float32", "int32"]:
                fill_val = df[col].median()
                df[col].fillna(fill_val, inplace=True)
                cleaning_steps.append(f"Filled '{col}' nulls with median ({fill_val:.2f})")
            elif role in ["categorical", "text"]:
                fill_val = df[col].mode()[0] if not df[col].mode().empty else "Unknown"
                df[col].fillna(fill_val, inplace=True)
                cleaning_steps.append(f"Filled '{col}' nulls with mode ('{fill_val}')")
        
        # ── Type casting ───────────────────────────────────────────────────────
        if role == "temporal" and "datetime" not in dtype:
            try:
                df[col] = pd.to_datetime(df[col], errors="coerce")
                cleaning_steps.append(f"Cast '{col}' to datetime")
            except Exception:
                pass
        
        # ── Outlier capping for numeric cols (IQR method) ─────────────────────
        if role == "numerical" and col in df.columns:
            try:
                Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
                IQR = Q3 - Q1
                lower, upper = Q1 - 3 * IQR, Q3 + 3 * IQR
                outlier_count = ((df[col] < lower) | (df[col] > upper)).sum()
                if outlier_count > 0:
                    df[col] = df[col].clip(lower, upper)
                    cleaning_steps.append(
                        f"Capped {outlier_count} outliers in '{col}' "
                        f"[{lower:.1f}, {upper:.1f}]"
                    )
            except Exception:
                pass
    
    log2 = f" [Cleaning Agent] Done. {len(cleaning_steps)} cleaning steps applied."
    print(log2)
    for step in cleaning_steps:
        print(f"   → {step}")
    
    return {
        **state,
        "cleaned_data": df,
        "agent_logs": state.get("agent_logs", []) + [log, log2] + cleaning_steps
    }

print("data_cleaning_agent defined ")


## 7. Agent 3 — Query Planning Agent
Translates natural language query → executable Pandas code.
Uses LLM with schema context for accurate code generation.


In [ ]:
def query_planning_agent(state: AnalystState) -> AnalystState:
    """
    Converts user's natural language question into a Pandas analysis plan.
    
    Responsibilities:
    - Understand user intent from query + schema context
    - Generate syntactically correct Pandas code
    - Store code string in state for Statistical Analysis Agent to execute
    """
    log = "📋 [Query Planning Agent] Generating analysis plan..."
    print(log)
    
    user_query:  str          = state["user_query"]
    schema_info: dict         = state["schema_info"]
    df:          pd.DataFrame = state["cleaned_data"]
    
    # Build column summary for the LLM
    col_summary = []
    for col, meta in schema_info["columns"].items():
        if col in df.columns:
            col_summary.append(
                f"  - {col}: {meta.get('role','?')} | dtype={meta['dtype']} | "
                f"nulls={meta['null_pct']}% | {meta.get('description','')}"
            )
    col_summary_str = "\n".join(col_summary)
    
    system_prompt = textwrap.dedent("""
        You are a Python data analyst. Given a user's question and a DataFrame schema,
        write executable Python/Pandas code to answer the question.
        
        Rules:
        1. The DataFrame is available as variable `df`
        2. Store ALL results in a dict called `results`
        3. results must contain: "summary" (str), "key_metrics" (dict), "data_for_viz" (dict)
        4. Do NOT use plt.show() or fig.show() — store figures separately if needed
        5. Handle edge cases (empty df, missing columns) gracefully
        6. Return ONLY the Python code block, no markdown fences, no explanation
    """)
    
    user_prompt = f"""
        User question: {user_query}
        
        DataFrame columns:
        {col_summary_str}
        
        DataFrame shape: {df.shape[0]} rows × {df.shape[1]} columns
        
        Write the Pandas analysis code:
    """
    
    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt)
    ])
    
    # Strip markdown fences if LLM adds them
    code = response.content.strip()
    if code.startswith("```"):
        lines = code.split("\n")
        code = "\n".join(lines[1:-1]) if lines[-1] == "```" else "\n".join(lines[1:])
    
    log2 = f" [Query Planning Agent] Generated {len(code.splitlines())} lines of analysis code."
    print(log2)
    print("\n── Generated Code Preview ──")
    print("\n".join(code.splitlines()[:15]))
    if len(code.splitlines()) > 15:
        print(f"   ... ({len(code.splitlines()) - 15} more lines)")
    
    return {
        **state,
        "query_plan": code,
        "agent_logs": state.get("agent_logs", []) + [log, log2]
    }

print("query_planning_agent defined ")


## 8. Agent 4 — Statistical Analysis Agent
Executes the query plan and adds supplementary EDA metrics.


In [ ]:
def statistical_analysis_agent(state: AnalystState) -> AnalystState:
    """
    Executes the Pandas code from Query Planning Agent.
    Also computes a standard EDA supplement (describe, correlations, value counts).
    
    Responsibilities:
    - Safe exec() of LLM-generated code in isolated namespace
    - Catch and log execution errors gracefully
    - Compute EDA stats as supplement
    - Store all analysis_results in state
    """
    log = "📊 [Statistical Analysis Agent] Executing analysis plan..."
    print(log)
    
    df:         pd.DataFrame = state["cleaned_data"]
    code:       str          = state["query_plan"]
    schema_info: dict        = state["schema_info"]
    
    analysis_results = {}
    errors = list(state.get("errors", []))
    
    # ── Step 1: Execute LLM-generated code ────────────────────────────────────
    exec_namespace = {"df": df.copy(), "pd": pd, "np": np, "results": {}}
    try:
        exec(code, exec_namespace)
        analysis_results["llm_results"] = exec_namespace.get("results", {})
        print("    LLM-generated code executed successfully")
    except Exception as e:
        err = f"Code execution error: {str(e)}"
        print(f"   ⚠️  {err}")
        errors.append(err)
        analysis_results["llm_results"] = {"error": str(e), "summary": "Code execution failed"}
    
    # ── Step 2: Supplement with standard EDA ──────────────────────────────────
    numeric_df = df.select_dtypes(include=[np.number])
    cat_cols   = [c for c, m in schema_info["columns"].items() 
                  if m.get("role") == "categorical" and c in df.columns]
    
    eda = {}
    
    # Descriptive statistics
    if not numeric_df.empty:
        eda["describe"] = numeric_df.describe().round(3).to_dict()
    
    # Correlation matrix (top correlated pairs)
    if numeric_df.shape[1] > 1:
        corr_matrix = numeric_df.corr()
        corr_pairs = []
        for i in range(len(corr_matrix.columns)):
            for j in range(i+1, len(corr_matrix.columns)):
                col_a = corr_matrix.columns[i]
                col_b = corr_matrix.columns[j]
                val   = corr_matrix.iloc[i, j]
                if not np.isnan(val):
                    corr_pairs.append({"col_a": col_a, "col_b": col_b, "correlation": round(val, 3)})
        corr_pairs.sort(key=lambda x: abs(x["correlation"]), reverse=True)
        eda["top_correlations"] = corr_pairs[:10]
    
    # Value counts for categoricals
    eda["value_counts"] = {}
    for col in cat_cols[:5]:  # Limit to 5 cat cols
        eda["value_counts"][col] = df[col].value_counts().head(10).to_dict()
    
    # Target variable summary
    target = schema_info.get("target_hint")
    if target and target in df.columns:
        eda["target_distribution"] = df[target].value_counts(normalize=True).round(3).to_dict()
    
    analysis_results["eda"] = eda
    
    log2 = (f" [Statistical Analysis Agent] Done. "
            f"EDA stats computed for {numeric_df.shape[1]} numeric cols, "
            f"{len(cat_cols)} categorical cols.")
    print(log2)
    
    return {
        **state,
        "analysis_results": analysis_results,
        "errors": errors,
        "agent_logs": state.get("agent_logs", []) + [log, log2]
    }

print("statistical_analysis_agent defined ")


## 9. Agent 5 — Visualization Agent
Generates Plotly charts dynamically based on analysis results + schema.
LLM decides which chart types are most appropriate.


In [ ]:
def visualization_agent(state: AnalystState) -> AnalystState:
    """
    Generates Plotly visualizations based on schema_info and analysis_results.
    
    Responsibilities:
    - Determine relevant chart types via LLM
    - Generate: distribution histograms, correlation heatmap,
                survival/target breakdown, categorical bar charts
    - Store list of Plotly figures in state
    """
    log = "📈 [Visualization Agent] Generating charts..."
    print(log)
    
    df:               pd.DataFrame = state["cleaned_data"]
    schema_info:      dict         = state["schema_info"]
    analysis_results: dict         = state["analysis_results"]
    user_query:       str          = state["user_query"]
    
    charts = []
    
    # ── Chart 1: Distribution of numerical columns ────────────────────────────
    num_cols = [c for c, m in schema_info["columns"].items() 
                if m.get("role") == "numerical" and c in df.columns 
                and m.get("analytical_importance") in ["high", "medium"]][:4]
    
    if num_cols:
        rows = (len(num_cols) + 1) // 2
        fig = make_subplots(rows=rows, cols=2,
                            subplot_titles=[f"Distribution: {c}" for c in num_cols])
        for i, col in enumerate(num_cols):
            row, col_idx = i // 2 + 1, i % 2 + 1
            fig.add_trace(
                go.Histogram(x=df[col], name=col, nbinsx=30,
                             marker_color="#636EFA", showlegend=False),
                row=row, col=col_idx
            )
        fig.update_layout(title_text="Numerical Feature Distributions",
                          height=300 * rows, template="plotly_white")
        charts.append({"title": "Numerical Distributions", "fig": fig})
        print(f"    Distribution chart: {num_cols}")
    
    # ── Chart 2: Correlation heatmap ──────────────────────────────────────────
    numeric_df = df.select_dtypes(include=[np.number])
    if numeric_df.shape[1] > 2:
        corr = numeric_df.corr().round(2)
        fig = go.Figure(go.Heatmap(
            z=corr.values, x=corr.columns.tolist(), y=corr.index.tolist(),
            colorscale="RdBu_r", zmid=0, text=corr.values.round(2),
            texttemplate="%{text}", colorbar_title="r"
        ))
        fig.update_layout(title="Feature Correlation Matrix",
                          height=500, template="plotly_white")
        charts.append({"title": "Correlation Heatmap", "fig": fig})
        print("    Correlation heatmap generated")
    
    # ── Chart 3: Target variable breakdown by categoricals ────────────────────
    target = schema_info.get("target_hint")
    cat_cols = [c for c, m in schema_info["columns"].items()
                if m.get("role") == "categorical" and c in df.columns
                and c != target and m.get("analytical_importance") == "high"][:3]
    
    if target and target in df.columns and cat_cols:
        fig = make_subplots(rows=1, cols=len(cat_cols),
                            subplot_titles=[f"{target} by {c}" for c in cat_cols])
        colors = ["#636EFA", "#EF553B", "#00CC96", "#AB63FA"]
        for i, col in enumerate(cat_cols):
            grouped = df.groupby(col)[target].mean().reset_index()
            fig.add_trace(
                go.Bar(x=grouped[col].astype(str), y=grouped[target],
                       name=col, marker_color=colors[i % len(colors)]),
                row=1, col=i + 1
            )
        fig.update_layout(title_text=f"{target} Rate by Categorical Features",
                          height=400, template="plotly_white", showlegend=False)
        charts.append({"title": f"{target} Rate by Category", "fig": fig})
        print(f"    Target breakdown chart: {target} by {cat_cols}")
    
    # ── Chart 4: Value counts for top categorical ─────────────────────────────
    vc_data = analysis_results.get("eda", {}).get("value_counts", {})
    if vc_data:
        top_cat = list(vc_data.keys())[0]
        counts  = vc_data[top_cat]
        fig = go.Figure(go.Bar(
            x=list(counts.keys()), y=list(counts.values()),
            marker_color="#00CC96", text=list(counts.values()),
            textposition="outside"
        ))
        fig.update_layout(title=f"Value Counts: {top_cat}",
                          height=400, template="plotly_white",
                          xaxis_title=top_cat, yaxis_title="Count")
        charts.append({"title": f"Value Counts: {top_cat}", "fig": fig})
        print(f"    Value count chart: {top_cat}")
    
    log2 = f" [Visualization Agent] Generated {len(charts)} charts."
    print(log2)
    
    return {
        **state,
        "charts": charts,
        "agent_logs": state.get("agent_logs", []) + [log, log2]
    }

print("visualization_agent defined ")


## 10. Agent 6 — Report Agent
Synthesizes all agent outputs into a structured markdown report using GPT-4o.


In [ ]:
def report_agent(state: AnalystState) -> AnalystState:
    """
    Final agent — synthesizes all outputs into a structured report.
    
    Responsibilities:
    - Summarize schema, cleaning steps, analysis results
    - Call GPT-4o to write the narrative insight section
    - Generate structured markdown report
    - Display charts inline
    """
    log = "📝 [Report Agent] Generating final report..."
    print(log)
    
    user_query:       str   = state["user_query"]
    schema_info:      dict  = state["schema_info"]
    analysis_results: dict  = state["analysis_results"]
    charts:           list  = state.get("charts", [])
    agent_logs:       list  = state.get("agent_logs", [])
    errors:           list  = state.get("errors", [])
    
    # ── Build context for LLM narrative ───────────────────────────────────────
    eda    = analysis_results.get("eda", {})
    llm_r  = analysis_results.get("llm_results", {})
    
    top_corrs = eda.get("top_correlations", [])[:5]
    corr_str  = "\n".join([f"  - {c['col_a']} ↔ {c['col_b']}: r={c['correlation']}" 
                             for c in top_corrs])
    
    target_dist = eda.get("target_distribution", {})
    
    context = f"""
        Dataset shape: {schema_info['shape']}
        User question: {user_query}
        Target variable: {schema_info.get('target_hint', 'none identified')}
        Target distribution: {json.dumps(target_dist, default=str)[:300]}
        Top correlations:
        {corr_str}
        LLM analysis summary: {str(llm_r.get('summary', ''))[:400]}
        Key metrics: {json.dumps(llm_r.get('key_metrics', {}), default=str)[:400]}
    """
    
    system_prompt = textwrap.dedent("""
        You are a senior data scientist writing a concise analysis report.
        Given dataset statistics, write a structured report with these sections:
        
        ## Executive Summary
        ## Key Findings (bullet points)
        ## Statistical Insights
        ## Recommendations
        
        Be specific with numbers. Be direct. Max 400 words.
    """)
    
    narrative_response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=context)
    ])
    
    # ── Assemble markdown report ───────────────────────────────────────────────
    report = f"""
# 📊 AI Data Analysis Report

**Query:** {user_query}  
**Dataset:** {schema_info['shape']['rows']} rows × {schema_info['shape']['cols']} columns  
**Agents Executed:** Schema → Cleaning → Query Planning → Statistical Analysis → Visualization → Report  

---

{narrative_response.content}

---

## 🔧 Agent Execution Log
{chr(10).join(['- ' + l for l in agent_logs[-10:]])}

## ⚠️ Errors Encountered
{chr(10).join(['- ' + e for e in errors]) if errors else '- None'}
    """
    
    # ── Display charts ─────────────────────────────────────────────────────────
    print("\n" + "="*60)
    print(report)
    print("="*60)
    
    for chart_data in charts:
        print(f"\n📊 Displaying: {chart_data['title']}")
        chart_data["fig"].show()
    
    log2 = " [Report Agent] Report generated successfully."
    print(f"\n{log2}")
    
    return {
        **state,
        "final_report": report,
        "agent_logs": agent_logs + [log, log2]
    }

print("report_agent defined ")


## 11. Supervisor — LangGraph Routing Logic
The supervisor decides which agent to call next using conditional edges.
This is the "brain" of the LangGraph StateGraph.


In [ ]:
def supervisor_router(state: AnalystState) -> str:
    """
    LangGraph conditional edge router.
    Returns the name of the next node to execute.
    
    Routing logic:
    schema_analysis → data_cleaning → query_planning 
    → statistical_analysis → visualization → report → END
    """
    # Determine which stage we're at by checking state completeness
    if state.get("schema_info") is None:
        return "schema_analysis"
    
    if state.get("cleaned_data") is None:
        return "data_cleaning"
    
    if state.get("query_plan") is None:
        return "query_planning"
    
    if state.get("analysis_results") is None:
        return "statistical_analysis"
    
    if state.get("charts") is None:
        return "visualization"
    
    if state.get("final_report") is None:
        return "report"
    
    return END

print("supervisor_router defined ")
print()
print("Routing table:")
print("  None → schema_analysis")
print("  schema_info ✓ → data_cleaning")
print("  cleaned_data ✓ → query_planning")
print("  query_plan ✓ → statistical_analysis")
print("  analysis_results ✓ → visualization")
print("  charts ✓ → report")
print("  final_report ✓ → END")


## 12. Build the LangGraph StateGraph
Wire all agents as nodes with the supervisor routing between them.


In [ ]:
def build_analyst_graph() -> StateGraph:
    """
    Assembles the full multi-agent LangGraph graph.
    
    Graph structure:
        START
          │
          ▼
       [supervisor] ──conditional──► schema_analysis
                                          │
                                     data_cleaning
                                          │
                                     query_planning
                                          │
                                   statistical_analysis
                                          │
                                     visualization
                                          │
                                        report
                                          │
                                         END
    """
    graph = StateGraph(AnalystState)
    
    # ── Add agent nodes ────────────────────────────────────────────────────────
    graph.add_node("schema_analysis",       schema_analysis_agent)
    graph.add_node("data_cleaning",         data_cleaning_agent)
    graph.add_node("query_planning",        query_planning_agent)
    graph.add_node("statistical_analysis",  statistical_analysis_agent)
    graph.add_node("visualization",         visualization_agent)
    graph.add_node("report",                report_agent)
    
    # ── Add supervisor node ────────────────────────────────────────────────────
    graph.add_node("supervisor", lambda state: state)  # Pass-through; routing is in edges
    
    # ── Entry point ────────────────────────────────────────────────────────────
    graph.set_entry_point("supervisor")
    
    # ── Conditional routing from supervisor ───────────────────────────────────
    graph.add_conditional_edges(
        "supervisor",
        supervisor_router,
        {
            "schema_analysis":      "schema_analysis",
            "data_cleaning":        "data_cleaning",
            "query_planning":       "query_planning",
            "statistical_analysis": "statistical_analysis",
            "visualization":        "visualization",
            "report":               "report",
            END:                    END
        }
    )
    
    # ── Each agent feeds back to supervisor for re-routing ─────────────────────
    for node in ["schema_analysis", "data_cleaning", "query_planning",
                 "statistical_analysis", "visualization", "report"]:
        graph.add_edge(node, "supervisor")
    
    return graph.compile()


analyst_graph = build_analyst_graph()
print(" LangGraph StateGraph compiled successfully!")
print()
print("Nodes:", ["supervisor", "schema_analysis", "data_cleaning",
                 "query_planning", "statistical_analysis", "visualization", "report"])


## 13. 🚀 Run End-to-End Analysis

**Query 1:** Survival rate analysis by gender, class, and age


In [ ]:
# ── Initialize state ──────────────────────────────────────────────────────────
initial_state: AnalystState = {
    "user_query":         "What factors most influenced survival on the Titanic? Analyze by gender, passenger class, and age.",
    "raw_data":           df_titanic,
    "file_path":          None,
    "schema_info":        None,
    "cleaned_data":       None,
    "query_plan":         None,
    "analysis_results":   None,
    "charts":             None,
    "next_agent":         None,
    "errors":             [],
    "agent_logs":         [],
    "final_report":       None
}

print("🚀 Starting AI Data Analyst Agent...")
print(f"Query: {initial_state['user_query']}")
print("="*60)

# ── Execute graph ──────────────────────────────────────────────────────────────
final_state = analyst_graph.invoke(initial_state)

print("\n" + "="*60)
print("🎉 Multi-Agent Analysis Complete!")
print(f"Total log entries: {len(final_state['agent_logs'])}")
print(f"Errors: {len(final_state['errors'])}")


## 14. Run a Second Query (same cleaned data, different question)
Demonstrates reusability — skip cleaning by passing cleaned_data directly.


In [ ]:
# Reuse cleaned data from previous run — skip schema + cleaning agents
initial_state_2: AnalystState = {
    "user_query":         "What was the average fare paid across different passenger classes? Show the distribution.",
    "raw_data":           df_titanic,
    "file_path":          None,
    "schema_info":        final_state["schema_info"],    # Reuse
    "cleaned_data":       final_state["cleaned_data"],   # Reuse
    "query_plan":         None,
    "analysis_results":   None,
    "charts":             None,
    "next_agent":         None,
    "errors":             [],
    "agent_logs":         ["[Reused] Schema and cleaning from previous run"],
    "final_report":       None
}

print("🚀 Running second query (schema + cleaning reused)...")
print(f"Query: {initial_state_2['user_query']}")
print("="*60)

final_state_2 = analyst_graph.invoke(initial_state_2)
print("\n Second analysis complete!")


## 15.  Notebook Complete — What's Next

### Phase 2: Architecture & Tech Stack Decision
- LangGraph vs AutoGen vs CrewAI for production
- FastAPI vs Streamlit for the interface layer
- GCP (Vertex AI + Cloud Run + BigQuery) vs Azure vs AWS

### Phase 3: OOP Modularization
```
ai_data_analyst/
├── agents/
│   ├── base_agent.py
│   ├── schema_agent.py
│   ├── cleaning_agent.py
│   ├── query_agent.py
│   ├── analysis_agent.py
│   ├── viz_agent.py
│   └── report_agent.py
├── graph/
│   ├── state.py
│   ├── graph_builder.py
│   └── router.py
├── data/
│   ├── loaders.py
│   └── validators.py
├── api/
│   └── main.py          ← FastAPI
├── Dockerfile
├── docker-compose.yml
└── .github/workflows/
    └── ci-cd.yml
```

### Phase 4: System Design
- Async agent execution with asyncio
- Redis for state persistence
- Celery for long-running analysis tasks
- Observability with LangSmith / OpenTelemetry

### Resume Talking Points (already earned from this notebook):
- "Built a 6-agent LangGraph system with typed state and conditional routing"
- "Implemented dynamic NL→Pandas code generation with safe exec() isolation"
- "Designed schema-aware cleaning pipeline with IQR outlier capping"
- "Generated Plotly dashboards programmatically based on LLM-inferred chart types"
